In [3]:
import pandas as pd
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

# 1. Daten laden
df = pd.read_csv("../data/ab_ag.tsv", sep="\t")
df_sars_cov2 = pd.read_csv("../data cleanup/pdb_ids_sars_cov2.csv")

# 2. Heavy Chains für jede PDB-ID aus df_sars_cov2 ermitteln
heavy_chain = [] 
for pdb_id in df_sars_cov2["pdb"]:
    heavy_chain_id = df.loc[df["pdb"] == pdb_id, "Hchain"].values
    if len(heavy_chain_id) > 0:
        heavy_chain.append(heavy_chain_id[0])
    else:
        heavy_chain.append(None)

# Neue Spalte anhängen
df_sars_cov2["heavy_chain"] = heavy_chain

df_sars_cov2

# 3. Funktion zur CDR-H3-Extraktion mit übergebener heavy_chain_id
def get_cdr_h_sequences(pdb_file:str, chain_id):
    """
    Extrahiert die CDR-H3-Sequenz aus der angegebenen PDB-Datei und Kette.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("pdb", pdb_file)

    cdr_h3_residues = []
    cdr_h2_residues = []
    cdr_h1_residues = []
    for model in structure:
        for chain in model:
            if chain.id == chain_id:
                for res in chain:
                    res_id = res.get_id()
                    if res_id[0] == " " and 95 <= res_id[1] <= 102:  # Chothia H3
                        cdr_h3_residues.append(res) 
                    elif res_id[0] == " " and 52 <= res_id[1] <= 56:  # Chothia H2
                        cdr_h2_residues.append(res)
                    elif res_id[0] == " " and 26 <= res_id[1] <= 32:  # Chothia H1
                        cdr_h1_residues.append(res)

    # Sequenz extrahieren
    aa_seq_h3 = ''.join(seq1(res["CA"].get_parent().resname) for res in cdr_h3_residues if "CA" in res)
    aa_seq_h2 = ''.join(seq1(res["CA"].get_parent().resname) for res in cdr_h2_residues if "CA" in res)
    aa_seq_h1 = ''.join(seq1(res["CA"].get_parent().resname) for res in cdr_h1_residues if "CA" in res)
    
    return aa_seq_h3, aa_seq_h2, aa_seq_h1

#Listen definieren, um die CDR-Sequenzen zu speichern
cdr_h3_seqs = []
cdr_h2_seqs = []
cdr_h1_seqs = []

for idx, row in df_sars_cov2.iterrows():
    pdb_id = row["pdb"]
    chain_id = row["heavy_chain"]
    
    pdb_file_path = f"../data/pdbs_corona_cnf/{pdb_id}.pdb"

    
    if os.path.exists(pdb_file_path) and chain_id:
        try:
            cdr_h3_seq, cdr_h2_seq, cdr_h1_seq = get_cdr_h_sequences(pdb_file_path, chain_id)
        except Exception as e:
            cdr_h3_seq = None
            cdr_h2_seq = None
            cdr_h1_seq = None
            print(f"Error for {pdb_id}: {e}")
    else:
        cdr_h3_seq = None
        cdr_h1_seq = None
        cdr_h2_seq = None

    cdr_h3_seqs.append(cdr_h3_seq)
    cdr_h2_seqs.append(cdr_h2_seq)
    cdr_h1_seqs.append(cdr_h1_seq)

    # 5. Als neue Spalte speichern und optional exportieren
df_sars_cov2["CDR_H3"] = cdr_h3_seqs
df_sars_cov2.to_csv("sars_cov2_cdrh3.tsv", sep="\t", index=False)
df_sars_cov2["CDR_H2"] = cdr_h2_seqs
df_sars_cov2.to_csv("sars_cov2_cdrh2.tsv", sep="\t", index=False)
df_sars_cov2["CDR_H1"] = cdr_h1_seqs
df_sars_cov2.to_csv("sars_cov2_cdrh1.tsv", sep="\t", index=False)

# Vorschau
print(df_sars_cov2.head())

df_sars_cov2




    pdb heavy_chain                 CDR_H3  CDR_H2     CDR_H1
0  9cci           B       TFGTYYDNTEDWFFDF  YGGDSD    GYSFSSF
1  9ccj           H              LPLGERIDY   YYSGT  GGSINTNMY
2  9bj2           H  HNGDPYDFWSGYNTWAGGLDV  IPFLDV    GVIFSRN
3  8z6r           E           QGDLGDWILLGY  YPGDSD    GYTFSYY
4  8z6s           E           QGDLGDWILLGY  YPGDSD    GYTFSYY


,pdb,heavy_chain,CDR_H3,CDR_H2,CDR_H1
0,9cci,B,TFGTYYDNTEDWFFDF,YGGDSD,GYSFSSF
1,9ccj,H,LPLGERIDY,YYSGT,GGSINTNMY
2,9bj2,H,HNGDPYDFWSGYNTWAGGLDV,IPFLDV,GVIFSRN
3,8z6r,E,QGDLGDWILLGY,YPGDSD,GYTFSYY
4,8z6s,E,QGDLGDWILLGY,YPGDSD,GYTFSYY
...,...,...,...,...,...
374,7tly,A,DYTRGAWFGESLIGGFDN,STYNGN,GYPFTSY
375,7chp,H,DLQEHGMDV,YSGGS,GFTVSSN
376,7n4i,H,LLYYSDSSPLDS,YPGDSD,GYSFISY
377,7l7d,H,PYCSSISCNDGFDI,VIGSGN,GFTFMSS


In [4]:
df_sars_cov2.to_csv("sars_cov2_cdr_seq.tsv", sep="\t", index=False)
